In [1]:
import sys
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

!{sys.executable} -m pip install matplotlib seaborn statsmodels geopandas plotly nbformat scikit-learn
!{sys.executable} -m pip install nbformat --upgrade

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.


In [2]:
op_data = pd.read_csv("/Users/nate/Desktop/8660-GroupProject/data/op_2021_data.csv", low_memory=False)

# clean and engineer key variables
op_data = op_data.rename(columns={
    "Specialty Description": "specialty",
    "Opioid Prescribing Rate": "opioid_rate",
    "Total Claim Count": "total_claims"
})

# drop missing values in key fields
op_data = op_data.dropna(subset=["opioid_rate", "specialty", "total_claims"])

# cap opioid rate to [0, 1] to avoid data entry errors
op_data = op_data[(op_data["opioid_rate"] >= 0) & (op_data["opioid_rate"] <= 1)]

# inspect structure
print(op_data[["specialty", "opioid_rate", "total_claims"]].head())


            specialty  opioid_rate  total_claims
0   Internal Medicine         0.03           492
1      Anesthesiology         0.49          1818
3  Nurse Practitioner         0.00           100
4     Family Practice         0.01          2766
6     General Surgery         0.29            41


In [3]:
# drop missing again just to be safe
model_data = op_data.dropna(subset=["opioid_rate", "specialty", "total_claims"])

# build model: opioid_rate ~ specialty + total_claims
model = smf.ols("opioid_rate ~ C(specialty) + total_claims", data=model_data).fit()

# print regression summary
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:            opioid_rate   R-squared:                       0.370
Model:                            OLS   Adj. R-squared:                  0.370
Method:                 Least Squares   F-statistic:                     2102.
Date:                Fri, 25 Apr 2025   Prob (F-statistic):               0.00
Time:                        05:39:51   Log-Likelihood:             4.8351e+05
No. Observations:              751560   AIC:                        -9.666e+05
Df Residuals:                  751349   BIC:                        -9.642e+05
Df Model:                         210                                         
Covariance Type:            nonrobust                                         
                                                                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------

In [4]:
ha = pd.read_csv("/Users/nate/HAEntity_full.csv")
computer = pd.read_csv("/Users/nate/Computer_full.csv")
pharmacy = pd.read_csv("/Users/nate/Pharmacy_full.csv")
pharmacy_prod = pd.read_csv("/Users/nate/PharmacyProduct_full.csv")
components = pd.read_csv("/Users/nate/UseOfITComponent_full.csv")
npi_map = pd.read_csv("/Users/nate/HAEntityNPI_full.csv")



/var/folders/5_/wt4fcyfx61761q_hmp3yxtjm0000gn/T/ipykernel_64068/1430958757.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  ha = pd.read_csv("/Users/nate/HAEntity_full.csv")


In [5]:
for name, df in {
    "HAEntity": ha,
    "Computer": computer,
    "Pharmacy": pharmacy,
    "PharmacyProduct": pharmacy_prod,
    "UseOfITComponent": components,
    "HAEntityNPI": npi_map,
}.items():
    print(f"\n{name} — shape: {df.shape}")
    print(df.columns.tolist())



HAEntity — shape: (65838, 57)
['HAEntityId', 'SurveyId', 'ParentId', 'UniqueId', 'EntityNo', 'Name', 'HAEntityTypeId', 'HAEntityType', 'CBSA', 'MedicareNumber', 'Address1', 'Address2', 'City', 'State', 'Zip', 'Phone', 'Website', 'Fax', 'EmailConvention', 'ProfitStatus', 'ServicePopulation', 'Type', 'YearOpened', 'OwnershipStatus', 'NofFTE', 'VendorSelStrategy', 'NofBeds', 'NofStaffedBeds', 'FreeStanding', 'SameISSystem', 'AcuteId', 'NofPhysicians', 'NofHCareVisits', 'FTETotal', 'DataCenterAcuteId', 'PhysFT', 'PhysAffiliated', 'PhysTotal', 'IsImaging', 'PhysResidents', 'PhysHospitalists', 'PhysOther', 'NofIntensiveCareBeds', 'AverageDailyCensus', 'Latitude', 'Longitude', 'IsIDSACOClassified', 'IsIDSPCMHCertified', 'IsIDSPlanningACO', 'IsIDSPlanningPCMH', 'NoOfNeonatalIntensiveCareBeds', 'PhysicianExtenders', 'PercVirtServers', 'PercVirtComputers', 'County', 'DisasterMsgId', 'IsNICUPresent']

Computer — shape: (15179, 11)
['Id', 'SurveyId', 'HAEntityId', 'ComputerType', 'VendorId', 'Ven

In [6]:
for name, df in {
    "HAEntity": ha,
    "Computer": computer,
    "Pharmacy": pharmacy,
    "PharmacyProduct": pharmacy_prod,
    "UseOfITComponent": components,
    "HAEntityNPI": npi_map,
}.items():
    print(f"\n{name} — Feature Nulls %")
    print(df.isnull().mean().sort_values(ascending=False).round(2))



HAEntity — Feature Nulls %
NofHCareVisits                   1.00
DisasterMsgId                    1.00
AverageDailyCensus               1.00
DataCenterAcuteId                1.00
PercVirtComputers                0.99
PercVirtServers                  0.99
PhysResidents                    0.98
PhysHospitalists                 0.98
PhysFT                           0.98
PhysOther                        0.98
PhysAffiliated                   0.98
FTETotal                         0.98
VendorSelStrategy                0.98
Fax                              0.97
AcuteId                          0.97
ServicePopulation                0.97
ProfitStatus                     0.97
PhysTotal                        0.94
NofFTE                           0.92
NofIntensiveCareBeds             0.92
NoOfNeonatalIntensiveCareBeds    0.92
EmailConvention                  0.90
Website                          0.89
NofBeds                          0.85
NofStaffedBeds                   0.85
MedicareNumber        

In [7]:
for name, df in {
    "HAEntity": ha,
    "Computer": computer,
    "Pharmacy": pharmacy,
    "PharmacyProduct": pharmacy_prod,
    "UseOfITComponent": components,
    "HAEntityNPI": npi_map,
}.items():
    print(f"\n{name} — first 3 rows")
    print(df.head(3))



HAEntity — first 3 rows
   HAEntityId  SurveyId  ParentId  UniqueId   EntityNo  \
0      856165     56048  856162.0     44570  100067299   
1      856166     56048  856162.0     27264      24193   
2      856168     56048  856162.0     27265      24195   

                                              Name  HAEntityTypeId  \
0  Behavioral Health at Marietta Memorial Hospital               2   
1                        The Rehabilitation Center               2   
2                       The Strecker Cancer Center               3   

  HAEntityType                                CBSA MedicareNumber  ...  \
0    Sub-Acute  Parkersburg-Marietta-Vienna, WV-OH            NaN  ...   
1    Sub-Acute  Parkersburg-Marietta-Vienna, WV-OH            NaN  ...   
2   Ambulatory  Parkersburg-Marietta-Vienna, WV-OH            NaN  ...   

  IsIDSPCMHCertified IsIDSPlanningACO IsIDSPlanningPCMH  \
0                  0                0                 0   
1                  0                0         

In [8]:
# aggregate number of computers per entity
computer_agg = computer.groupby("HAEntityId")["NofComputers"].sum().reset_index()

# pick pharmacy binary tech features
pharm_cols = ["HAEntityId", "Robot", "ADM", "Carousels", "InventoryMgmt"]
pharm_clean = pharmacy[pharm_cols].copy()

# pivot UseOfITComponent to count component types per HAEntityId
component_counts = components.groupby(["HAEntityId", "Component"]).size().unstack(fill_value=0).reset_index()

# start from HAEntity
df = ha[["HAEntityId", "NofBeds", "ProfitStatus", "NofPhysicians"]].copy()

# merge everything
df = df.merge(computer_agg, on="HAEntityId", how="left")
df = df.merge(pharm_clean, on="HAEntityId", how="left")
df = df.merge(component_counts, on="HAEntityId", how="left")

# drop rows without target variables
df = df[df["NofBeds"].notnull() & df["ProfitStatus"].notnull()]


In [9]:
# fill missing tech usage with zeros
df[["NofComputers", "Robot", "ADM", "Carousels", "InventoryMgmt"]] = (
    df[["NofComputers", "Robot", "ADM", "Carousels", "InventoryMgmt"]].fillna(0).astype(int)
)

# convert ProfitStatus to binary
df["is_profit"] = df["ProfitStatus"].apply(lambda x: int(str(x).strip().lower() == "for profit"))

# confirm
print(df.head())


     HAEntityId  NofBeds    ProfitStatus  NofPhysicians  NofComputers  Robot  \
136      856192   6612.0  Not For Profit            NaN             0      0   
576      856772     25.0  Not For Profit            NaN             0      0   
582      856777     54.0  Not For Profit            NaN             0      0   
587      856781     25.0  Not For Profit            NaN             0      0   
590      856784     32.0  Not For Profit            NaN             0      0   

     ADM  Carousels  InventoryMgmt  Access to Diagnostic Tests  Bill Payment  \
136    0          0              0                         NaN           NaN   
576    0          0              0                         NaN           NaN   
582    0          0              0                         NaN           NaN   
587    0          0              0                         NaN           NaN   
590    0          0              0                         NaN           NaN   

     For Clinical Documentation Charti

In [10]:
component_cols = df.columns.difference(["HAEntityId", "NofBeds", "ProfitStatus", "NofPhysicians", "NofComputers", "Robot", "ADM", "Carousels", "InventoryMgmt", "is_profit"])
df[component_cols] = df[component_cols].fillna(0).astype(int)


In [11]:
# predictors
X_ols = df[["NofComputers", "Robot", "ADM", "Carousels", "InventoryMgmt"] + list(component_cols)]
X_ols = sm.add_constant(X_ols)
y_ols = df["NofBeds"]

# fit model
ols_model = sm.OLS(y_ols, X_ols).fit()
print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:                NofBeds   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                       nan
Date:                Fri, 25 Apr 2025   Prob (F-statistic):                nan
Time:                        05:39:52   Log-Likelihood:                -18032.
No. Observations:                2007   AIC:                         3.607e+04
Df Residuals:                    2006   BIC:                         3.607e+04
Df Model:                           0                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
co

/Users/nate/Library/Python/3.9/lib/python/site-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_log = df[["NofComputers", "Robot", "ADM", "Carousels", "InventoryMgmt"] + list(component_cols)]
y_log = df["is_profit"]

X_train, X_test, y_train, y_test = train_test_split(X_log, y_log, stratify=y_log, random_state=42)

clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.93      1.00      0.96       467
           1       0.00      0.00      0.00        35

    accuracy                           0.93       502
   macro avg       0.47      0.50      0.48       502
weighted avg       0.87      0.93      0.90       502



/Users/nate/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/nate/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/nate/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [13]:
op = pd.read_csv("/Users/nate/Desktop/8660-GroupProject/data/op_2021_data.csv")
op = op.rename(columns={"NPPES Provider ZIP Code": "Zip"})

# standardize ZIPs to 5 digits
op["Zip"] = op["Zip"].astype(str).str[:5]
ha["Zip"] = ha["Zip"].astype(str).str[:5]

# aggregate opioid rate per ZIP
zip_avg = op.groupby("Zip")["Opioid Prescribing Rate"].mean().reset_index()
zip_avg.columns = ["Zip", "avg_opioid_rate"]

# merge with HAEntity
df_with_opioids = ha.merge(zip_avg, on="Zip", how="left")
print(df_with_opioids[["HAEntityId", "Zip", "avg_opioid_rate"]].head())


   HAEntityId    Zip  avg_opioid_rate
0      856165  45750         0.096322
1      856166  45750         0.096322
2      856168  45750         0.096322
3      856169  45750         0.096322
4      856170  45750         0.096322


In [14]:
# start by bringing in ZIP into df from ha
df = df.merge(ha[["HAEntityId", "Zip"]], on="HAEntityId", how="left")

# then merge opioid ZIP averages
df = df.merge(zip_avg, on="Zip", how="left")


In [15]:
X = df[["avg_opioid_rate", "NofComputers", "Robot", "ADM", "Carousels", "is_profit"]].fillna(0)
X = sm.add_constant(X)
y = df["NofBeds"]

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                NofBeds   R-squared:                       0.031
Model:                            OLS   Adj. R-squared:                  0.030
Method:                 Least Squares   F-statistic:                     31.75
Date:                Fri, 25 Apr 2025   Prob (F-statistic):           2.65e-14
Time:                        05:39:55   Log-Likelihood:                -18001.
No. Observations:                2007   AIC:                         3.601e+04
Df Residuals:                    2004   BIC:                         3.602e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const             153.0119     84.644     

/Users/nate/Library/Python/3.9/lib/python/site-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])


In [16]:
X = df[["avg_opioid_rate", "NofComputers", "Robot", "ADM", "Carousels"]].fillna(0)
y = df["is_profit"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.93      1.00      0.96       467
           1       0.00      0.00      0.00        35

    accuracy                           0.93       502
   macro avg       0.47      0.50      0.48       502
weighted avg       0.87      0.93      0.90       502



/Users/nate/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/nate/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/nate/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [17]:
df["opioid_bin"] = pd.qcut(df["avg_opioid_rate"], q=4, labels=["Low", "Med-Low", "Med-High", "High"])
df.groupby("opioid_bin")[["NofComputers", "Robot", "ADM", "Carousels"]].mean()
# aggregate HIMSS signals to ZIP level
zip_factors = df.groupby("Zip").agg({
    "is_profit": "mean",
    "NofComputers": "mean",
    "Robot": "mean",
    "ADM": "mean",
    "Carousels": "mean"
}).reset_index()

# merge with ZIP-level opioid prescribing
zip_model = zip_factors.merge(zip_avg, on="Zip", how="inner")

# regression
X = sm.add_constant(zip_model[["is_profit", "NofComputers", "Robot", "ADM", "Carousels"]])
y = zip_model["avg_opioid_rate"]

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:        avg_opioid_rate   R-squared:                         nan
Model:                            OLS   Adj. R-squared:                    nan
Method:                 Least Squares   F-statistic:                       nan
Date:                Fri, 25 Apr 2025   Prob (F-statistic):                nan
Time:                        05:39:56   Log-Likelihood:                    nan
No. Observations:                1808   AIC:                               nan
Df Residuals:                    1806   BIC:                               nan
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const               nan        nan        nan   

/var/folders/5_/wt4fcyfx61761q_hmp3yxtjm0000gn/T/ipykernel_64068/3454706662.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("opioid_bin")[["NofComputers", "Robot", "ADM", "Carousels"]].mean()
/Users/nate/Library/Python/3.9/lib/python/site-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])


In [18]:
# prepare opioid dataset
op = pd.read_csv("/Users/nate/Desktop/8660-GroupProject/data/op_2021_data.csv")
op = op.rename(columns={"NPPES Provider ZIP Code": "Zip"})

# clean ZIP
op["Zip"] = op["Zip"].astype(str).str[:5]

# join ZIP-level HIMSS data
op_model = op.merge(zip_factors, on="Zip", how="left")

# drop rows w/ null opioid prescribing rate
op_model = op_model[op_model["Opioid Prescribing Rate"].notnull()]

# encode specialty
op_model["specialty"] = op_model["Specialty Description"].astype("category")

# regression
import statsmodels.formula.api as smf
model = smf.ols("Q('Opioid Prescribing Rate') ~ C(specialty) + is_profit + Robot + ADM + NofComputers", data=op_model).fit()
print(model.summary())


                                 OLS Regression Results                                 
Dep. Variable:     Q('Opioid Prescribing Rate')   R-squared:                       0.356
Model:                                      OLS   Adj. R-squared:                  0.356
Method:                           Least Squares   F-statistic:                     698.1
Date:                          Fri, 25 Apr 2025   Prob (F-statistic):               0.00
Time:                                  05:40:05   Log-Likelihood:             1.1119e+05
No. Observations:                        194544   AIC:                        -2.221e+05
Df Residuals:                            194389   BIC:                        -2.205e+05
Df Model:                                   154                                         
Covariance Type:                      nonrobust                                         
                                                                                                  coef    std 

In [19]:
op = pd.read_csv("/Users/nate/Desktop/8660-GroupProject/data/op_2021_data.csv")
op = op.rename(columns={"NPPES Provider ZIP Code": "Zip"})
op["Zip"] = op["Zip"].astype(str).str[:5]

ha = pd.read_csv("/Users/nate/HAEntity_full.csv")
ha["Zip"] = ha["Zip"].astype(str).str[:5]

zip_avg = op.groupby("Zip")["Opioid Prescribing Rate"].mean().reset_index()
zip_avg.columns = ["Zip", "avg_opioid_rate"]

df = ha.merge(zip_avg, on="Zip", how="left")


/var/folders/5_/wt4fcyfx61761q_hmp3yxtjm0000gn/T/ipykernel_64068/4202514114.py:5: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  ha = pd.read_csv("/Users/nate/HAEntity_full.csv")


In [28]:
predictors = ["NofPhysicians", "NofIntensiveCareBeds"]


df[predictors + ["avg_opioid_rate"]].isna().sum()


NofPhysicians           20364
NofIntensiveCareBeds    60567
avg_opioid_rate          6311
dtype: int64

In [29]:
for col in ["NofPhysicians", "NofIntensiveCareBeds"]:
    subdf = df[[col, "avg_opioid_rate"]].dropna()
    if len(subdf) < 100:
        print(f"Skipping {col}: not enough data")
        continue
    X = sm.add_constant(subdf[[col]])
    y = subdf["avg_opioid_rate"]
    model = sm.OLS(y, X).fit()
    print(f"\n=== {col.upper()} ===")
    print(model.summary())



=== NOFPHYSICIANS ===
                            OLS Regression Results                            
Dep. Variable:        avg_opioid_rate   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     11.54
Date:                Fri, 25 Apr 2025   Prob (F-statistic):           0.000682
Time:                        05:43:05   Log-Likelihood:                 71225.
No. Observations:               40693   AIC:                        -1.424e+05
Df Residuals:                   40691   BIC:                        -1.424e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.088